<a href="https://github.com/gutris1/segsmaker">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)

In [ ]:
# @title WebUI Installer {"display-mode":"form"}
# @markdown ### Step 1 - Choose WebUI
Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown If you publish Repo Fusion to your own GitHub repository, change this raw base URL once here.
Repo_Raw_Base = 'https://github.com/gutris1/segsmaker/raw/main' # @param {type:"string", placeholder:"https://github.com/OWNER/REPO/raw/main"}
# @markdown ---
# @markdown ### Step 2 - API Keys
# @markdown Get Civitai key: https://civitai.com/user/account
Civitai___Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
# @markdown Get Hugging Face token: https://huggingface.co/settings/tokens
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Hugging Face READ token (optional)"}
# @markdown ---
# @markdown ### Step 3 - Google Drive
Mount__GDrive = 'No' # @param ["Yes", "No"]

import subprocess
import sys
from pathlib import Path

if Mount__GDrive == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

_setup_py = '/content/setup.py'
_repo_raw_base = Repo_Raw_Base.strip().rstrip('/') or 'https://github.com/gutris1/segsmaker/raw/main'
_setup_url = f'{_repo_raw_base}/script/KC/setup.py'
_result = subprocess.run(['curl', '-fLo', _setup_py, _setup_url], capture_output=True, text=True)
if _result.returncode != 0:
    print(f'Setup script download failed:\n{_result.stderr}')
    sys.exit(1)

print('Setup script downloaded. Running installer...')
get_ipython().run_line_magic(
    'run',
    f'{_setup_py} --webui="{Webui}" --civitai_key="{Civitai___Key}" --hf_read_token="{HF_Read_Token}" --repo_raw_base="{_repo_raw_base}"'
)

if Mount__GDrive == 'Yes':
    drive_root = Path('/content/drive/MyDrive/Segsmaker')

    for name, target in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        persistent_dir = drive_root / name
        persistent_dir.mkdir(parents=True, exist_ok=True)
        link_path = target / f'drive-{name}'
        if not link_path.exists():
            link_path.symlink_to(persistent_dir, target_is_directory=True)

    get_ipython().system(f'rm -rf "{WebUI_Output}"')
    output_dir = drive_root / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    output_dir.mkdir(parents=True, exist_ok=True)
    if not WebUI_Output.exists():
        WebUI_Output.symlink_to(output_dir, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        cache_link = WebUI / 'cache'
        get_ipython().system(f'rm -rf "{cache_link}"')
        cache_dir = drive_root / 'cache'
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_link.symlink_to(cache_dir, target_is_directory=True)

## Model Downloader

In [ ]:
# @title Model Downloader - 5 Checkpoint + 5 LoRA + VAE {"display-mode":"form"}
# @markdown ### Checkpoints
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### LoRA
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### VAE
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### Speed Options
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download


def _download_one(url, destination, filename=None):
    command = f'{url} {destination}'
    if filename:
        command = f'{command} {filename}'
    get_ipython().run_line_magic('download', command)


_queue = []
for _url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if _url.strip():
        _queue.append((_url.strip(), str(CKPT), None))
for _url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if _url.strip():
        _queue.append((_url.strip(), str(LORA), None))
if VAE_URL.strip():
    _queue.append((VAE_URL.strip(), str(VAE), None))

if not _queue:
    print('No model URLs provided; skipping.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    for _url, _destination, _filename in _queue:
        _download_one(_url, _destination, _filename)

## Extra Assets

In [ ]:
# @title Extra Assets - Extensions, Embeddings, Upscalers {"display-mode":"form"}
# @markdown ### Extensions / ComfyUI Custom Nodes
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### Embeddings
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### Upscalers
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### Speed Options
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import shlex
import subprocess

from nenen88 import parallel_batch_download


def _normalize_clone_entry(entry):
    entry = entry.strip()
    if entry.startswith('git clone '):
        entry = entry[len('git clone '):].strip()
    return entry


def _clone_extension(entry):
    parts = shlex.split(_normalize_clone_entry(entry))
    if not parts:
        return ('empty', False)

    repo_url = parts[0]
    folder_name = parts[1] if len(parts) > 1 else None
    display_name = folder_name or Path(repo_url.rstrip('/')).name.replace('.git', '')
    command = ['git', 'clone', '--depth=1', repo_url]
    if folder_name:
        command.append(folder_name)

    try:
        result = subprocess.run(
            command,
            cwd=str(Extensions),
            capture_output=True,
            text=True,
            timeout=900,
        )
        if result.returncode == 0:
            return (display_name, True)

        fallback = ['git', 'clone', repo_url]
        if folder_name:
            fallback.append(folder_name)
        retry = subprocess.run(
            fallback,
            cwd=str(Extensions),
            capture_output=True,
            text=True,
            timeout=900,
        )
        return (display_name, retry.returncode == 0)
    except Exception:
        return (display_name, False)


_extension_urls = [
    _url.strip()
    for _url in [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
    if _url.strip()
]

if _extension_urls:
    Path(Extensions).mkdir(parents=True, exist_ok=True)
    print(f'Cloning {len(_extension_urls)} extension/custom-node repositories...')
    if Assets_Parallel_Download and len(_extension_urls) > 1:
        with ThreadPoolExecutor(max_workers=Assets_Max_Workers) as _pool:
            _futures = [_pool.submit(_clone_extension, _entry) for _entry in _extension_urls]
            for _index, _future in enumerate(as_completed(_futures), start=1):
                _name, _ok = _future.result()
                _mark = 'OK' if _ok else 'FAILED'
                print(f'[{_index}/{len(_extension_urls)}] {_mark} {_name}')
    else:
        for _index, _entry in enumerate(_extension_urls, start=1):
            _name, _ok = _clone_extension(_entry)
            _mark = 'OK' if _ok else 'FAILED'
            print(f'[{_index}/{len(_extension_urls)}] {_mark} {_name}')

_asset_queue = []
for _url in [Embedding_1, Embedding_2, Embedding_3]:
    if _url.strip():
        _asset_queue.append((_url.strip(), str(Embeddings), None))
for _url in [Upscaler_1, Upscaler_2, Upscaler_3]:
    if _url.strip():
        _asset_queue.append((_url.strip(), str(Upscalers), None))

if _asset_queue:
    if Assets_Parallel_Download:
        parallel_batch_download(_asset_queue, max_workers=Assets_Max_Workers)
    else:
        for _url, _destination, _filename in _asset_queue:
            command = f'{_url} {_destination}'
            if _filename:
                command = f'{command} {_filename}'
            get_ipython().run_line_magic('download', command)
else:
    print('No embedding/upscaler URLs provided; skipping file assets.')

## FLUX Models

In [ ]:
# @title FLUX Model Downloader {"display-mode":"form"}
# @markdown ### FLUX Variant
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown ---
# @markdown ### Component URLs
FLUX_Unet = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
# @markdown ---
# @markdown ### Speed Options
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:4, step:1}

from nenen88 import parallel_batch_download

if FLUX_Variant == 'None':
    print('FLUX_Variant is None; skipping.')
else:
    _unet_url = FLUX_Unet
    if 'dev' in FLUX_Variant.lower() and 'schnell' in _unet_url:
        _unet_url = _unet_url.replace('schnell', 'dev')

    _flux_queue = [
        (_unet_url, str(UNET), None),
        (FLUX_Clip_L, str(CLIP), None),
        (FLUX_T5XXL, str(CLIP), None),
        (FLUX_VAE, str(VAE), 'flux_ae.safetensors'),
    ]
    _flux_queue = [(_url, _destination, _filename) for _url, _destination, _filename in _flux_queue if _url.strip()]

    if not _flux_queue:
        print('No FLUX URLs provided; skipping.')
    elif Parallel_FLUX_Download:
        print(f'Downloading {FLUX_Variant} with {len(_flux_queue)} files in parallel...')
        parallel_batch_download(_flux_queue, max_workers=FLUX_Max_Workers)
    else:
        for _url, _destination, _filename in _flux_queue:
            command = f'{_url} {_destination}'
            if _filename:
                command = f'{command} {_filename}'
            get_ipython().run_line_magic('download', command)

## Temporary Models

In [ ]:
# @title Temporary Model Downloader {"display-mode":"form"}
# @markdown Temporary files are stored in runtime storage and are lost after session reset.
TMP_Checkpoint_1 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Checkpoint_2 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Lora_1 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Lora_2 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Parallel_Download = True # @param {type:"boolean"}
TMP_Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download

_tmp_queue = []
for _raw, _destination in [
    (TMP_Checkpoint_1, TMP_CKPT),
    (TMP_Checkpoint_2, TMP_CKPT),
    (TMP_Lora_1, TMP_LORA),
    (TMP_Lora_2, TMP_LORA),
]:
    if not _raw.strip():
        continue
    _parts = _raw.strip().split(None, 1)
    _tmp_queue.append((_parts[0], str(_destination), _parts[1] if len(_parts) > 1 else None))

if not _tmp_queue:
    print('No temporary URLs provided; skipping.')
elif TMP_Parallel_Download:
    parallel_batch_download(_tmp_queue, max_workers=TMP_Max_Workers)
else:
    for _url, _destination, _filename in _tmp_queue:
        command = f'{_url} {_destination}'
        if _filename:
            command = f'{command} {_filename}'
        get_ipython().run_line_magic('download', command)

## Launch

In [ ]:
# @title Launcher WebUI {"display-mode":"form"}
# @markdown Select the same WebUI that you installed in the first cell.
Software = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown Tunnel tokens are optional and are never printed by this cell.
Ngrok_Token = '' # @param {type:"string", placeholder:"Optional ngrok token"}
Zrok_Token = '' # @param {type:"string", placeholder:"Optional zrok token"}
# @markdown ---
Extra_Args = '' # @param {type:"string", placeholder:"Optional extra launch arguments"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

args_map = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

selected_args = args_map.get(Software, '').strip()
if Extra_Args.strip():
    selected_args = f'{selected_args} {Extra_Args.strip()}'.strip()
if Skip_ComfyUI_Check:
    selected_args = f'{selected_args} --skip-comfyui-check'.strip()
if Skip_Widget:
    selected_args = f'{selected_args} --skip-widget'.strip()

display_args = selected_args
if Ngrok_Token.strip():
    selected_args = f'{selected_args} --N={Ngrok_Token.strip()}'.strip()
    display_args = f'{display_args} --N=<hidden>'.strip()
if Zrok_Token.strip():
    selected_args = f'{selected_args} --Z={Zrok_Token.strip()}'.strip()
    display_args = f'{display_args} --Z=<hidden>'.strip()

if 'Webui' in globals() and Software != Webui:
    print(f'Warning: installed WebUI is {Webui}, but launcher selection is {Software}.')
print(f'Launching {Software} with args: {display_args}')

get_ipython().run_line_magic('cd', '-q $WebUI')
get_ipython().run_line_magic('run', f'segsmaker.py {selected_args}')